In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor


In [3]:
#Upload data file
df=pd.read_json('SafeWalk_Dummy_DatabaseML.json')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   _id               60 non-null     object 
 1   locationId        60 non-null     object 
 2   coordinates       60 non-null     object 
 3   lighting          60 non-null     int64  
 4   cctvCount         60 non-null     int64  
 5   crimeIndex        60 non-null     float64
 6   crowdDensity      60 non-null     float64
 7   policeDistance    60 non-null     float64
 8   hospitalDistance  60 non-null     float64
dtypes: float64(4), int64(2), object(3)
memory usage: 4.3+ KB


In [4]:
#drop firs column
df.drop('_id',axis=1)

,locationId,coordinates,lighting,cctvCount,crimeIndex,crowdDensity,policeDistance,hospitalDistance
0,Shivajinagar,"[18.5302, 73.8526]",5,15,0.10,0.80,0.4,0.8
1,FC-Road,"[18.5293, 73.848]",5,14,0.09,0.90,0.6,1.0
2,JM-Road,"[18.5276, 73.8471]",4,12,0.12,0.85,0.7,1.2
3,Viman-Nagar,"[18.5667, 73.9145]",5,20,0.08,0.90,0.8,1.0
4,Koregaon-Park,"[18.5362, 73.895]",5,18,0.10,0.85,1.1,1.5
5,Magarpatta,"[18.5196, 73.9317]",5,25,0.07,0.70,1.4,1.6
6,Hinjewadi-Phase1,"[18.591, 73.7389]",4,14,0.10,0.75,2.0,2.5
7,Baner,"[18.559, 73.7867]",4,16,0.12,0.80,1.5,1.7
8,Aundh,"[18.557, 73.8068]",5,17,0.08,0.85,1.1,1.2
9,Balewadi-HighStreet,"[18.5642, 73.7708]",4,15,0.09,0.80,1.8,2.0


In [5]:
#Convert coordinates array into seperate columns
df["lat"] = df["coordinates"].apply(lambda x: x[0])
df["lon"] = df["coordinates"].apply(lambda x: x[1])
df = df.drop("coordinates", axis=1)
df=df.drop('_id',axis=1)
df.sample(5)


,locationId,lighting,cctvCount,crimeIndex,crowdDensity,policeDistance,hospitalDistance,lat,lon
53,Gokhale-Nagar,1,2,0.70,0.60,2.0,2.4,18.5292,73.8283
42,Shukrawar-Peth,1,3,0.70,0.70,1.8,2.0,18.5100,73.8550
43,Nana-Peth,2,3,0.68,0.65,1.9,2.2,18.5186,73.8683
35,Thergaon,3,7,0.29,0.60,2.4,2.2,18.6001,73.7861
47,Bhosari,2,4,0.65,0.60,3.0,2.8,18.6342,73.8493


In [6]:
#train_test_split
X = df[["lighting", "cctvCount", "crowdDensity", "policeDistance", "hospitalDistance", "lat", "lon"]]
y = df["crimeIndex"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [7]:
#train model using Random forest
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200, random_state=42)

In [8]:
#pridict crime index
df["predicted_crimeIndex"] = model.predict(X)

In [9]:
#calculate risk Score
df["riskScore"] = df["predicted_crimeIndex"] * df["crowdDensity"]


In [10]:
print(df[["locationId", "predicted_crimeIndex", "riskScore"]].head())

      locationId  predicted_crimeIndex  riskScore
0   Shivajinagar               0.09960   0.079680
1        FC-Road               0.09775   0.087975
2        JM-Road               0.11695   0.099407
3    Viman-Nagar               0.08460   0.076140
4  Koregaon-Park               0.10180   0.086530


In [16]:
df.to_json("safetyData.json",orient="records",indent=2)